## Função que padroniza a chamada dos scripts, facilitando a captura de erros e documentação dos resultados


In [43]:
import subprocess
import time
import shutil

# Criamos uma lista vazia no topo do seu script para armazenar os resultados de cada tarefa
historico_execucoes = []

def executar_limpeza(nome_tarefa, comando_ps):
    print(f"[{nome_tarefa}] Iniciando...")
    espaco_antes = shutil.disk_usage("C:\\").free # 1. Mede o espaço livre no disco C: ANTES da limpeza (em Bytes)
    inicio = time.time()
    
    resultado = subprocess.run( # 2. Executa o comando PowerShell
        ["powershell", "-Command", comando_ps],
        capture_output=True,
        text=True
    )
    
    fim = time.time()
    tempo_execucao = fim - inicio
    espaco_depois = shutil.disk_usage("C:\\").free # 3. Mede o espaço livre no disco C: DEPOIS da limpeza (em Bytes)

    # Variáveis padrão
    sucesso = (resultado.returncode == 0)
    mb_liberados = 0.0
    erros_ignorados = ""
    erro_critico = ""
    
    if sucesso:
        bytes_liberados = espaco_depois - espaco_antes
        mb_liberados = bytes_liberados / (1024 * 1024) if bytes_liberados > 0 else 0.0
            
        print(f"[{nome_tarefa}] Concluído em {tempo_execucao:.2f} segundos.")
        print(f" └─ Espaço liberado: {mb_liberados:.2f} MB")
        
        if resultado.stdout.strip():
            erros_ignorados = resultado.stdout.strip()
            print(" └─ [Aviso] Alguns arquivos estavam bloqueados. Relatório guardado para o PDF.\n")
    else:
        erro_critico = resultado.stderr.strip()
        print(f"[{nome_tarefa}] Falha. Erro: {erro_critico}\n")

    #empacotamos todos os dados coletados em um dicionário
    dados = {
        "tarefa": nome_tarefa,
        "sucesso": sucesso,
        "tempo": tempo_execucao,
        "mb_liberados": mb_liberados,
        "erros_ignorados": erros_ignorados,
        "erro_critico": erro_critico
    }
    
    #salvamos o dicionário na nossa lista global
    historico_execucoes.append(dados)

## Função para gerar relatórios dinamicamente

In [64]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY

def gerar_relatorio_pdf():
    pdf_nome = r"C:\Users\Barco de Papel Dev\Desktop\Relatorio_Automacao_Dinamico.pdf"
    documento = SimpleDocTemplate(pdf_nome, pagesize=A4, rightMargin=50, leftMargin=50, topMargin=50, bottomMargin=50)

    estilos = getSampleStyleSheet()
    estilo_titulo = estilos['Heading1']
    estilo_subtitulo = estilos['Heading2']
    
    estilo_texto = ParagraphStyle('Justificado', parent=estilos['Normal'], alignment=TA_JUSTIFY, spaceAfter=12, fontSize=11)
    
    # Estilo específico para a lista de scripts (com um pouco mais de espaçamento e recuo)
    estilo_lista = ParagraphStyle('Lista', parent=estilos['Normal'], alignment=TA_JUSTIFY, spaceAfter=14, fontSize=11, leftIndent=10)
    
    estilo_codigo = estilos['Code']
    estilo_codigo.fontSize = 8
    estilo_codigo.backColor = '#f4f4f4'
    estilo_codigo.spaceAfter = 12

    conteudo = []
    
    # TÍTULO PRINCIPAL
    conteudo.append(Paragraph("Relatório Técnico: Automação e Manutenção Preventiva do Windows", estilo_titulo))
    conteudo.append(Spacer(1, 10))
    
    # TÓPICO 1
    conteudo.append(Paragraph("1. Objetivo e Justificativa", estilo_subtitulo))
    texto1 = "Este projeto tem como objetivo central automatizar a rotina de manutenção preventiva do sistema operacional Windows. A iniciativa busca substituir processos de limpeza manuais, frequentemente suscetíveis a falhas e esquecimentos, por uma arquitetura programada e rastreável. Ao desenvolvermos uma solução própria, eliminamos a dependência de softwares de terceiros (que muitas vezes operam como 'caixas pretas'), assegurando total transparência sobre os dados manipulados e preservando a integridade do ambiente corporativo."
    conteudo.append(Paragraph(texto1, estilo_texto))
    
    # TÓPICO 2
    conteudo.append(Paragraph("2. Metodologia e Tecnologias Utilizadas", estilo_subtitulo))
    texto2 = "O desenvolvimento foi conduzido no ambiente Jupyter Notebook, adotando a linguagem Python como orquestradora de processos. A comunicação de baixo nível com o sistema operacional foi estabelecida através da biblioteca nativa <b>subprocess</b>, permitindo a execução de comandos PowerShell de forma invisível e a captura controlada de logs de erro. Adicionalmente, empregaram-se as bibliotecas <b>shutil</b>, para o cálculo preciso das métricas de I/O (espaço liberado em disco), e <b>time</b>, para o benchmarking de execução. A documentação final é gerada de forma automatizada e imutável via <b>ReportLab</b>."
    conteudo.append(Paragraph(texto2, estilo_texto))
    
    # TÓPICO 3
    conteudo.append(Paragraph("3. Escopo de Atuação e Segurança", estilo_subtitulo))
    texto3 = "Para mitigar riscos de instabilidade do sistema operacional nesta fase inicial, a rotina foi desenhada sob um modelo seguro, atuando estritamente no escopo do usuário atual (dispensando a elevação de privilégios administrativos). Os módulos implementados focam na higienização de arquivos altamente voláteis: remoção de resíduos de instalação (Pasta Temp), exclusão do cache fragmentado gerado por navegadores (Chrome e Edge) e limpeza do histórico de atalhos recentes. O script foi projetado para ignorar arquivos atualmente em uso, evitando o corrompimento de processos ativos."
    conteudo.append(Paragraph(texto3, estilo_texto))

    # TÓPICO 4 - CATÁLOGO DE SCRIPTS (NOVO)
    conteudo.append(Paragraph("4. Catálogo de Scripts Implementados", estilo_subtitulo))
    conteudo.append(Paragraph("Abaixo estão catalogados todos os módulos de limpeza desenvolvidos neste projeto, detalhando suas funções e os privilégios necessários para execução:", estilo_texto))
    
    # Lista de Scripts
    script1 = "<b>1. Esvaziamento da Lixeira:</b><br/><i>Função:</i> Limpa todos os arquivos previamente deletados em todas as unidades de disco locais. Serve para recuperar espaço superficial de forma rápida.<br/><i>Privilégio Necessário:</i> Usuário Comum (Não requer Administrador)."
    conteudo.append(Paragraph(script1, estilo_lista))
    
    script2 = "<b>2. Limpeza da Pasta Temp do Usuário:</b><br/><i>Função:</i> Apaga arquivos residuais e temporários gerados pelas aplicações durante a sessão do usuário atual (ex: logs e instaladores extraídos).<br/><i>Privilégio Necessário:</i> Usuário Comum (Não requer Administrador)."
    conteudo.append(Paragraph(script2, estilo_lista))

    script3 = "<b>3. Limpeza do Histórico de Arquivos Recentes:</b><br/><i>Função:</i> Remove os atalhos invisíveis que o Windows Explorer gera para alimentar a aba de 'Acesso Rápido'. Não deleta os arquivos originais, apenas limpa o rastro de navegação.<br/><i>Privilégio Necessário:</i> Usuário Comum (Não requer Administrador)."
    conteudo.append(Paragraph(script3, estilo_lista))

    script4 = "<b>4. Limpeza de Cache de Navegadores (Chrome e Edge):</b><br/><i>Função:</i> Varre os diretórios de `User Data` dos navegadores para excluir milhares de arquivos de cache de sites. Exige que os navegadores estejam fechados para eficácia total.<br/><i>Privilégio Necessário:</i> Usuário Comum (Não requer Administrador)."
    conteudo.append(Paragraph(script4, estilo_lista))

    script5 = "<b>5. Limpeza do Temp do Sistema e Prefetch:</b><br/><i>Função:</i> Higieniza a pasta raiz do Windows Temp e a pasta Prefetch (que armazena dados de pré-carregamento de softwares). Remove resíduos globais do sistema operacional.<br/><i>Privilégio Necessário:</i> <font color='red'>Requer Administrador</font>."
    conteudo.append(Paragraph(script5, estilo_lista))

    script6 = "<b>6. Limpeza do Cache do Windows Update (SoftwareDistribution):</b><br/><i>Função:</i> Interrompe temporariamente os serviços de rede, esvazia os gigabytes de atualizações e patches antigos baixados pelo Windows e reativa os serviços.<br/><i>Privilégio Necessário:</i> <font color='red'>Requer Administrador</font>."
    conteudo.append(Paragraph(script6, estilo_lista))

    script7 = "<b>7. Limpeza Profunda de Componentes (WinSxS / DISM):</b><br/><i>Função:</i> Aciona a ferramenta nativa DISM para analisar e remover versões obsoletas de componentes vitais do Windows substituídos por novas atualizações. É uma operação de alto nível de I/O.<br/><i>Privilégio Necessário:</i> <font color='red'>Requer Administrador</font>."
    conteudo.append(Paragraph(script7, estilo_lista))

    script8 = "<b>8. Limpeza de Logs de Eventos (Event Viewer):</b><br/><i>Função:</i> Varre e esvazia todos os registros históricos (erros, avisos, logs de segurança) acumulados pelo Visualizador de Eventos do sistema.<br/><i>Privilégio Necessário:</i> <font color='red'>Requer Administrador</font>."
    conteudo.append(Paragraph(script8, estilo_lista))

    # TÓPICO 5 (Antigo Tópico 4)
    conteudo.append(Paragraph("5. Resultados Operacionais da Sessão", estilo_subtitulo))
    
    for item in historico_execucoes:
        conteudo.append(Paragraph(f"<b>Módulo Executado:</b> {item['tarefa']}", estilo_texto))
        
        if item['sucesso']:
            resumo = f"• Tempo de execução: {item['tempo']:.2f} segundos<br/>• Armazenamento recuperado: {item['mb_liberados']:.2f} MB"
            conteudo.append(Paragraph(resumo, estilo_texto))
            
            if item['erros_ignorados']:
                conteudo.append(Paragraph("<i>Exceções Tratadas (Arquivos bloqueados por processos do sistema):</i>", estilo_texto))
                conteudo.append(Preformatted(item['erros_ignorados'], estilo_codigo))
        else:
            conteudo.append(Paragraph("• Status: FALHA CRÍTICA", estilo_texto))
            conteudo.append(Preformatted(item['erro_critico'], estilo_codigo))
            
        conteudo.append(Spacer(1, 10))

    # GERAR ARQUIVO
    documento.build(conteudo)
    print(f"\n✅ Relatório gerado com sucesso! Arquivo salvo como: {pdf_nome}")

### 1. **Limpar a Pasta Temp do Usuário**
Remove arquivos temporários gerados pelas aplicações executadas com o perfil do usuário atual. É uma operação superficial e não afeta o sistema global.
Tempo de espera: Muito Rápido. Apenas leitura e exclusão de arquivos pequenos sem bloqueios de sistema.

In [45]:
cmd_temp_user = r'Remove-Item -Path "$env:TEMP\*" -Recurse -Force -ErrorAction SilentlyContinue; exit 0'
executar_limpeza("Limpeza Temp do Usuário", cmd_temp_user)

[Limpeza Temp do Usuário] Iniciando...
[Limpeza Temp do Usuário] Concluído em 0.56 segundos.
 └─ Espaço liberado: 0.00 MB


In [41]:
#essa versão mostra o que e por que não pode ser apagado
cmd_temp_user = r'''
# 1. Tenta apagar, esconde os erros nativos, mas guarda na variável 'falhas'
Remove-Item -Path "$env:TEMP\*" -Recurse -Force -ErrorAction SilentlyContinue -ErrorVariable falhas

# 2. Se a variável 'falhas' tiver algo, listamos o que aconteceu
if ($falhas) {
    foreach ($erro in $falhas) {
        # Imprime o caminho do arquivo e a mensagem descritiva do erro
        Write-Output "-> Nao apagou: $($erro.TargetObject)"
        Write-Output "   Motivo: $($erro.Exception.Message)"
    }
}

# 3. Força o sucesso para o Python
exit 0
'''

executar_limpeza("Limpeza Temp do Usuário", cmd_temp_user)

[Limpeza Temp do Usuário] Iniciando...
[Limpeza Temp do Usuário] Concluído em 0.54 segundos.
 └─ Espaço liberado: 0.00 MB

 └─ Relatório de Arquivos Ignorados:
-> Nao apagou: _unins-done.tmp
   Motivo: Acesso negado ao caminho.
-> Nao apagou: _unins.tmp
   Motivo: Acesso negado ao caminho.
-> Nao apagou: C:\Users\BARCOD~1\AppData\Local\Temp\is-HHDBWN1WPM-uninstall.tmp
   Motivo: Acesso negado ao caminho.
-> Nao apagou: C:\Users\BARCOD~1\AppData\Local\Temp\15bc2004-7973-410d-864f-66298db64954.tmp
   Motivo: O processo nÆo pode acessar o arquivo 'C:\Users\BARCOD~1\AppData\Local\Temp\15bc2004-7973-410d-864f-66298db64954.tmp' porque ele est  sendo usado por outro processo.
-> Nao apagou: C:\Users\BARCOD~1\AppData\Local\Temp\238b7d76-2f70-4aab-8bef-e96bc40d6207.tmp
   Motivo: O processo nÆo pode acessar o arquivo 'C:\Users\BARCOD~1\AppData\Local\Temp\238b7d76-2f70-4aab-8bef-e96bc40d6207.tmp' porque ele est  sendo usado por outro processo.
-> Nao apagou: C:\Users\BARCOD~1\AppData\Local\Temp\

### 2. **Limpar o Histórico de Arquivos Recentes**
Limpa os atalhos (.lnk) gerados pelo Windows Explorer que alimentam a aba "Acesso Rápido". Não exclui os arquivos originais.
Tempo de espera: Instantâneo. O volume de dados (em KB) é minúsculo.

In [46]:
cmd_recentes = r'Remove-Item -Path "$env:APPDATA\Microsoft\Windows\Recent\*" -Recurse -Force -ErrorAction SilentlyContinue'
executar_limpeza("Limpeza de Arquivos Recentes", cmd_recentes)

[Limpeza de Arquivos Recentes] Iniciando...
[Limpeza de Arquivos Recentes] Concluído em 0.45 segundos.
 └─ Espaço liberado: 0.02 MB


### 3. **Esvaziar a Lixeira**
Utiliza o cmdlet nativo Clear-RecycleBin para esvaziar a lixeira de todas as unidades locais. A flag -Force suprime o prompt de confirmação do Windows.
Tempo de espera: Rápido a Moderado. Depende estritamente do volume (GB) e da quantidade de itens que estavam acumulados na lixeira.

In [47]:
cmd_lixeira = r'Clear-RecycleBin -Force -ErrorAction SilentlyContinue; exit 0'
executar_limpeza("Esvaziar Lixeira", cmd_lixeira)

[Esvaziar Lixeira] Iniciando...
[Esvaziar Lixeira] Concluído em 0.45 segundos.
 └─ Espaço liberado: 0.00 MB


### 4. **Limpar Cache de Navegadores**
Navegadores fragmentam o cache em milhares de pequenos arquivos em disco. O script visa os diretórios padrão do Chrome e do Edge. 
Nota técnica: O navegador deve estar fechado, caso contrário, alguns arquivos estarão travados em processo (lock).
Tempo de espera: Moderado. Devido à grande quantidade de arquivos (I/O intensivo), mesmo em SSDs rápidos, leva alguns segundos a mais que pastas normais.

In [48]:
cmd_cache_browsers = r'''
$chrome = "$env:LOCALAPPDATA\Google\Chrome\User Data\Default\Cache\*"
$edge = "$env:LOCALAPPDATA\Microsoft\Edge\User Data\Default\Cache\*"
Remove-Item -Path $chrome -Recurse -Force -ErrorAction SilentlyContinue; 
Remove-Item -Path $edge -Recurse -Force -ErrorAction SilentlyContinue; exit 0
'''
executar_limpeza("Limpeza de Cache (Chrome/Edge)", cmd_cache_browsers)

[Limpeza de Cache (Chrome/Edge)] Iniciando...
[Limpeza de Cache (Chrome/Edge)] Concluído em 0.74 segundos.
 └─ Espaço liberado: 23.96 MB


In [36]:
#essa versão mostra o que e por que não pode ser apagado
cmd_cache_browsers = r'''
$chrome = "$env:LOCALAPPDATA\Google\Chrome\User Data\Default\Cache\*"
$edge = "$env:LOCALAPPDATA\Microsoft\Edge\User Data\Default\Cache\*"

# 1. Tenta limpar o Chrome e guarda os erros em 'falhasChrome'
Remove-Item -Path $chrome -Recurse -Force -ErrorAction SilentlyContinue -ErrorVariable falhasChrome

# 2. Tenta limpar o Edge e guarda os erros em 'falhasEdge'
Remove-Item -Path $edge -Recurse -Force -ErrorAction SilentlyContinue -ErrorVariable falhasEdge

# 3. Junta as duas listas de erros em uma só para facilitar a leitura
$todasFalhas = @()
if ($falhasChrome) { $todasFalhas += $falhasChrome }
if ($falhasEdge) { $todasFalhas += $falhasEdge }

# 4. Se houver falhas, exibe o relatório detalhado
if ($todasFalhas) {
    foreach ($erro in $todasFalhas) {
        Write-Output "-> Nao apagou: $($erro.TargetObject)"
        Write-Output "   Motivo: $($erro.Exception.Message)"
    }
}

# 5. Só agora finalizamos com sucesso para o Python
exit 0
'''

executar_limpeza("Limpeza de Cache (Chrome/Edge)", cmd_cache_browsers)

[Limpeza de Cache (Chrome/Edge)] Iniciando...
[Limpeza de Cache (Chrome/Edge)] Concluído em 0.60 segundos.
 └─ Espaço liberado: 0.00 MB

 └─ Relatório de Arquivos Ignorados:
-> Nao apagou: data_0
   Motivo: O acesso ao caminho 'data_0' foi negado.
-> Nao apagou: data_1
   Motivo: O acesso ao caminho 'data_1' foi negado.
-> Nao apagou: data_2
   Motivo: O acesso ao caminho 'data_2' foi negado.
-> Nao apagou: data_3
   Motivo: O acesso ao caminho 'data_3' foi negado.
-> Nao apagou: index
   Motivo: O acesso ao caminho 'index' foi negado.
-> Nao apagou: C:\Users\Barco de Papel Dev\AppData\Local\Microsoft\Edge\User Data\Default\Cache\Cache_Data
   Motivo: A pasta nÆo est  vazia.

-> Nao apagou: journal.baj
   Motivo: O processo nÆo pode acessar o arquivo 'journal.baj' porque ele est  sendo usado por outro processo.
-> Nao apagou: C:\Users\Barco de Papel Dev\AppData\Local\Microsoft\Edge\User Data\Default\Cache\No_Vary_Search
   Motivo: A pasta nÆo est  vazia.
-------------------------------

### **5. Limpeza do Cache do Windows Update (SoftwareDistribution)**
O Windows não apaga automaticamente os arquivos de instalação de atualizações antigas. Com o tempo, essa pasta pode acumular gigabytes de dados inúteis. Como o serviço do Windows Update "tranca" essa pasta, o script precisa parar o serviço, apagar os arquivos e religar o serviço.
Tempo de espera: Moderado a Demorado (5 a 20 segundos, dependendo de quanto tempo o serviço demora para parar).

In [58]:
cmd_win_update = r'''
# 1. Para os serviços de atualização
Stop-Service -Name wuauserv -Force -ErrorAction SilentlyContinue
Stop-Service -Name bits -Force -ErrorAction SilentlyContinue

# 2. Apaga o cache de downloads do Windows Update
$update_path = "$env:windir\SoftwareDistribution\Download\*"
Remove-Item -Path $update_path -Recurse -Force -ErrorAction SilentlyContinue -ErrorVariable falhas

# 3. Religa os serviços
Start-Service -Name wuauserv -ErrorAction SilentlyContinue
Start-Service -Name bits -ErrorAction SilentlyContinue

if ($falhas) {
    foreach ($erro in $falhas) {
        Write-Output "-> Nao apagou: $($erro.TargetObject)"
        Write-Output "   Motivo: $($erro.Exception.Message)"
    }
}
exit 0
'''
executar_limpeza("Limpeza do Windows Update", cmd_win_update)

[Limpeza do Windows Update] Iniciando...
[Limpeza do Windows Update] Concluído em 0.48 segundos.
 └─ Espaço liberado: 0.00 MB
 └─ [Aviso] Alguns arquivos estavam bloqueados. Relatório guardado para o PDF.



## **6. Limpeza Profunda de Componentes (WinSxS / DISM)**
A pasta WinSxS (Component Store) guarda backups de arquivos de sistema para o caso de você querer desinstalar uma atualização no futuro. A ferramenta nativa DISM analisa e remove as versões de atualizações que já foram substituídas por versões mais novas.
Tempo de espera: Muito Demorado (Pode levar de 1 a 10 minutos). Esse é um processo de I/O pesado que analisa dependências do sistema. O Python vai ficar "congelado" aguardando o fim da operação.

In [59]:
cmd_dism = r'''
# O DISM é um executável nativo. O parâmetro /StartComponentCleanup faz a higienização
Write-Output "Iniciando análise profunda do DISM. Isso pode demorar vários minutos..."
DISM.exe /Online /Cleanup-Image /StartComponentCleanup

# Como o DISM lida com seus próprios erros, apenas forçamos a saída bem-sucedida para o Python
exit 0
'''
executar_limpeza("Limpeza de Componentes do Sistema (DISM)", cmd_dism)

[Limpeza de Componentes do Sistema (DISM)] Iniciando...
[Limpeza de Componentes do Sistema (DISM)] Concluído em 0.46 segundos.
 └─ Espaço liberado: 0.01 MB
 └─ [Aviso] Alguns arquivos estavam bloqueados. Relatório guardado para o PDF.



## **7. Pastas Temp do Sistema e Prefetch**
O Prefetch guarda fragmentos de inicialização de programas. Com o tempo, ele armazena dados de softwares que você já desinstalou. O Temp do sistema guarda logs de instalações globais (C:\Windows\Temp).
Tempo de espera: Rápido (1 a 3 segundos).



In [60]:
cmd_sys_temp_prefetch = r'''
$sys_temp = "$env:windir\Temp\*"
$prefetch = "$env:windir\Prefetch\*"

Remove-Item -Path $sys_temp -Recurse -Force -ErrorAction SilentlyContinue -ErrorVariable falhasTemp
Remove-Item -Path $prefetch -Recurse -Force -ErrorAction SilentlyContinue -ErrorVariable falhasPrefetch

$todasFalhas = @()
if ($falhasTemp) { $todasFalhas += $falhasTemp }
if ($falhasPrefetch) { $todasFalhas += $falhasPrefetch }

if ($todasFalhas) {
    foreach ($erro in $todasFalhas) {
        Write-Output "-> Nao apagou: $($erro.TargetObject)"
        Write-Output "   Motivo: $($erro.Exception.Message)"
    }
}
exit 0
'''
executar_limpeza("Limpeza Temp do Sistema e Prefetch", cmd_sys_temp_prefetch)

[Limpeza Temp do Sistema e Prefetch] Iniciando...
[Limpeza Temp do Sistema e Prefetch] Concluído em 0.44 segundos.
 └─ Espaço liberado: 0.00 MB


## **8. Limpeza de Logs de Eventos do Windows (Event Viewer)**
O Windows registra absolutamente tudo o que acontece (alertas, erros de hardware, tentativas de login) em arquivos de log espalhados pelo "Visualizador de Eventos". Em máquinas antigas, esses registros antigos ocupam espaço desnecessário e dificultam auditorias novas.
Tempo de espera: Moderado (5 a 15 segundos). Ele percorre dezenas de categorias de log do sistema e esvazia uma por uma.

In [61]:
cmd_event_logs = r'''
# Coleta todos os logs do sistema e os limpa usando o wevtutil
wevtutil el | Foreach-Object {wevtutil cl "$_" -ErrorAction SilentlyContinue}

# Esse comando geralmente é silencioso, mas retorna erro apenas se um log crítico do kernel estiver travado.
Write-Output "Todos os logs de eventos não-críticos foram limpos."
exit 0
'''
executar_limpeza("Limpeza de Logs de Eventos (Event Viewer)", cmd_event_logs)

[Limpeza de Logs de Eventos (Event Viewer)] Iniciando...
[Limpeza de Logs de Eventos (Event Viewer)] Concluído em 18.89 segundos.
 └─ Espaço liberado: 0.00 MB
 └─ [Aviso] Alguns arquivos estavam bloqueados. Relatório guardado para o PDF.



### **Gerador de Relatório PDF**
Script para gerar relaório PDF.

In [65]:
gerar_relatorio_pdf()


✅ Relatório gerado com sucesso! Arquivo salvo como: C:\Users\Barco de Papel Dev\Desktop\Relatorio_Automacao_Dinamico.pdf
